# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified relaxation buoyancy field and surface wind stress.
twnh 

Dec '25   Write `ExampleSolution_v0.5.ipynb` <br>
Jul '26.  Write `FrictionalGeostrophicEquationSolution.ipynb`

This notebook solves the non-dimensional, linear, time-independent frictional geostrophic equations:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + Ek \,  \frac{d}{dz} \left( \nu \frac{d u}{d z} \right), \\
  f  u & = - \frac{\partial p}{\partial y} + Ek \,  \frac{d}{dz} \left( \nu \frac{d v}{d z} \right) , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu (z)$ (customized so $\nu = 1$ at the moment), and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \nu \partial_z u & = \tau^x \\
\displaystyle  \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the applied surface stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This notebook also derives the equation satisfied by the surface pressure field $p_s(x,y)$.

In [ ]:
using SymPy

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ = symbols("z ξ", real=true, negative=true)   # Vertical coordinate and source location (both in [-H,0])
x, y = symbols("x y", real=true)                  # Horizontal coordinates
H = symbols("H", real=true, positive=true)        # Domain depth
geometry_params = (H, z, ξ)

Ek = symbols("Ek ", real=true, positive=true)     # Ekman number

# Frictional geostrophic equations symbolic parameters:
f = symbols("f", real=true, positive=true)        # Coriolis parameter
τs = SymFunction("τs", complex=true)              # Complex surface stress
psg = SymFunction("psg", complex=true)            # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)

# Buoyancy symbolic parameters:
b = SymFunction("b", real=true)                   # Buoyancy field b(x,y,z)
ψ = symbols("ψ", real=true, positive=true)        # Diffusivity parameters

# Define viscosity profile here:
ν = symbols("ν", real=true, positive=true)        # Viscosity parameters
uv_params = (f, Ek, ν)

# Convenience variable:
φ = symbols("φ", imaginary=true)                  # Main solution parameter (φ = ϕ√i)
f_subs = φ^2 * Ek * ν / im

### Compute pressure fields for the specified buoyancy field $b$:

In [ ]:
# Define buoyancy gradient field:
bg = SymFunction("bg", complex=true)                # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x, y, ξ), (ξ, z, 0))    # Baroclinic presure gradient 
pg = psg(x, y) + pbarog                             # Total pressure gradient
pbotg = pg.subs(z, -H(x, y))                        # Bottom pressure gradient

### Function to solve the frictional geostrophic equation using a Green's function: This is where `SymPy` is unavoidable as there's no `Symbolics` equivalent to `dsolve` as of July 2026.

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, Ek, ν = uv_params
    H, z, ξ = geometry_params
    uv = SymFunction("uv")
    A = symbols("A", real=true)                                                         # Unknown coefficient in the Green's function solution

    # 0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + Ek * diff(ν * diff(uv(z), z), z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    Gₘ = dsolve(ode, uv(z), ics=Dict(uv(z).subs(z, -H)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z), Gₘ)) == 0                                # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1], A)
    @assert sympy.simplify(Gₘ.subs(z, -H)) == 0                                         # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics=Dict(diff(uv(z), z).subs(z, 0)=>0)).rhs
    @assert simplify(expand(ode.lhs.subs(uv(z), Gₚ)).doit()) == 0                       # Check solution
    @assert sympy.simplify(diff(Gₚ, z).subs(z, 0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z, ξ) / (ν.subs(z, ξ) * W.subs(z, ξ))
    Gp = Gₘ.subs(z, ξ) * Gₚ / (ν.subs(z, ξ) * W.subs(z, ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z, ξ) - Gp.subs(z, ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z, ξ) - diff(Gp, z).subs(z, ξ)) + 1/(ν.subs(z, ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z, ξ)), (Gp, Ge(z, ξ)))

    # Check boundary conditions:
    @assert sympy.simplify(diff(G, z).subs(z, 0).subs(ξ, -H//2)) == 0
    @assert sympy.simplify(G.subs(z, -H).subs(ξ, -H//2)) == 0

    # Final sympy.simplify (to cancel constants). Avoid sympy.simplify in general because it's not always reproducible.
    G = sympy.simplify(G)
    return G
end;

### Compute the velocity Green's function:

In [ ]:
Guv_sym = compute_Guv(uv_params, geometry_params)
Guv = simplify(Guv_sym.subs(f, f_subs))                          # This is where \varphi is defined.
Guv = simplify(Guv.subs(sqrt(- φ^2), φ * im))                    # Substitute the square root of -φ^2: amounts to choice of branch cut for the square root function.
println("1.1 Symbolic computation of Guv completed.")
display(Guv)

# Save the expression string to a text file
open("sympy_Guv_expression.txt", "w") do io
    write(io, string(Guv))
end
println("Wrote sympy_Guv_expression.txt file.")

In [ ]:
# Used for stress-driven part of the velocity:
Guv0 = Guv.subs(ξ, 0)
Latex_Guv0 = (exp(-φ * z)/(ν * φ * (1 + exp(2 * φ * H)))) * (1 - exp(2 * φ * (z + H)))
resid = Guv0 - Latex_Guv0
@assert resid == 0
println("1.2 LaTeX Guv0 expression matches.")
display(Latex_Guv0)

# Used for surface-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv), (ξ, -H, 0)).args[1].args[1]
max_obj = sympy.Max(z, -H)
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
                      Latex_Guv_int_wrt_ξ = (1 / (ν * φ^2 * (1 + exp(2 * φ * H)))) *
                      (-1 - exp(2 * φ * H) + exp(φ * (z + H)) + exp(φ * (H - z)))
resid = Guv_int_wrt_ξ - Latex_Guv_int_wrt_ξ
@assert resid == 0
println("1.3 LaTeX Guv integral wrt ξ expression matches.")
display(Latex_Guv_int_wrt_ξ)

In [ ]:
# Used for bottom boundary stress computation:
tmp = diff(Guv, z)
tmp2 = tmp.subs(z, -H)
dGuv_dz_at_bottom = tmp2.args[1].args[1]
Latex_dGuv_dz_at_bottom = (-exp(-φ * (ξ - H))) / (ν * (1 + exp(2 * φ * H))) * (1 + exp(2 * φ * ξ))
resid = dGuv_dz_at_bottom - Latex_dGuv_dz_at_bottom
@assert resid == 0
println("1.4 LaTeX d/dz Guv(z,ξ) @ z = -H matches.")
display(Latex_dGuv_dz_at_bottom)

Latex_dGuv_dz_at_bottom_ξ0 = Latex_dGuv_dz_at_bottom.subs(ξ, 0)
resid = dGuv_dz_at_bottom.subs(ξ, 0) - Latex_dGuv_dz_at_bottom_ξ0
@assert resid == 0
println("1.5 LaTeX d/dz Guv(z,0) @ z = -H matches.")
display(Latex_dGuv_dz_at_bottom_ξ0)

### Symbolic computation of flow $u(z),v(z), \tau_b, U, V$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(sympy.expand(Guv * (1/Ek) * pg.subs(z, ξ)), (ξ, -H, 0))
𝔲1 = 𝔲1.subs(max_obj, z)             # Buoyancy-driven part of the velocity
𝔲1 = simplify(𝔲1)
𝔲2 = - Guv0 * τs(x, y)               # Stress-driven part of the velocity
𝔲2 = simplify(𝔲2)
𝔲 = 𝔲1 + 𝔲2
println("2.1 Symbolic computation of 𝔲 completed.")
display(𝔲)

In [ ]:
# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + Ek * diff(ν * diff(𝔲1, z), z)
tmp1 = simplify(tmp1.subs(f, f_subs))
tmp2 = -im * f * 𝔲2 + Ek * diff(ν * diff(𝔲2, z), z)
tmp2 = simplify(tmp2.subs(f, f_subs))
@assert tmp1 + tmp2 == pg
println("2.2 Final expression for 𝔲 satisfies the original differential equation.")

In [ ]:
# Compute bottom stress on fluid:
τb = - ν * diff(𝔲, z).subs(z, -H)

# Check expression for surface stress on fluid:
@assert sympy.simplify(diff(𝔲1, z).subs(z, 0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ν * diff(𝔲, z).subs(z, 0)
@assert sympy.simplify(tmp - τs(x, y)) == 0
println("2.3 Expression for surface stress on fluid is correct.")

In [ ]:
# Check integral of surface pressure term for bottom stress:
tmp_τ_integrand = sympy.expand((1 + exp(2*φ*ξ)) * exp(-φ*(ξ-H)) / (Ek * (1 + exp(2*φ*H))))
tmp = sympy.simplify(sympy.integrate(tmp_τ_integrand, (ξ, -H, 0)).args[1].args[1])
Latex_S_fn = (1/(φ * Ek)) * sympy.tanh(φ*H)

# Check S(x) function:
XXX = symbols("XXX")                            # SymPy can't pull the constant psg factor outside the integral.
τb_tmp = sympy.expand(τb.subs(psg(x, y), XXX).doit())
psg_coeff = τb_tmp.coeff(XXX)
testS = sympy.simplify(Latex_S_fn - sympy.simplify(psg_coeff))
@assert testS == 0
println("2.5 LaTeX S(x) function matches.")
display(Latex_S_fn)

# Check T(x) function:
function make_Latex_T_fn(int_bg_term)
    tmp_T_integrand2 = sympy.expand(tmp_τ_integrand * int_bg_term)
    tmp_T_fn = (sympy.integrate(tmp_T_integrand2, (ξ, -H, 0)) - τs(x, y)*(2*exp(φ*H))/(1 + exp(2*φ*H)))
    return tmp_T_fn
end
int_bg_term = sympy.integrate(bg(x, y, ξ), (ξ, ξ, 0))        # Generic integral of bg.
Latex_T_fn = make_Latex_T_fn(int_bg_term)
τb_tmp2 = (τb_tmp - psg_coeff * XXX).doit()
testT = τb_tmp2 - Latex_T_fn
YYY = symbols("YYY")                            # SymPy can't work on the buried bg integral, so substitute for it
testT = testT.subs(bg(x, y, ξ), YYY)
testT = sympy.expand(testT).doit()
@assert testT == 0
println("2.6 LaTeX T(x) function matches.")
display(Latex_T_fn)

In [ ]:
# Compute depth-integrated flow:
𝔘 = SymPy.integrate(sympy.expand(𝔲), (z, -H, 0))

# Check A(x) function:
Latex_A_fn = (1/(ν * φ^2 * Ek )) *  ( (exp(2*φ*H) - 1) / (φ * (exp(2*φ*H) + 1)) - H  )
𝔘_tmp = sympy.expand(𝔘.subs(psg(x, y), XXX).doit())
psg_coeff = 𝔘_tmp.coeff(XXX)
testA = sympy.simplify(Latex_A_fn - psg_coeff)
@assert testA == 0
println("2.5 LaTeX A(x) function matches.")
display(Latex_A_fn)

# Check alternative form of A(x) function:
Latex_A_fn_alt = H/ (φ^2 * Ek ) * ( (sympy.tanh(φ*H) / (φ*H)) - 1 )
testA_alt = sympy.simplify(Latex_A_fn - Latex_A_fn_alt)
@assert testA_alt == 0
println("2.5b Alternative form of LaTeX A(x) function matches.")
display(Latex_A_fn_alt)

In [ ]:
# Check integral of surface pressure term for 𝔘:
tmp_ps_integrand = sympy.expand(exp(-φ*ξ)*(exp(φ*ξ) - exp(φ*H))*(exp(φ*(ξ + H)) - 1))
tmp = sympy.integrate(tmp_ps_integrand, (ξ, -H, 0))
Latex_tmp = - H * (exp(2*φ*H) + 1) + (1/φ)*(exp(2*φ*H) - 1)
@assert tmp == Latex_tmp
println("2.6 LaTeX integral of the 𝔘 surface pressure term matches.")

In [ ]:
# Check B(x) function:
function make_Latex_B_fn(int_bg_term)
    tmp_B_integrand2 = sympy.expand(tmp_ps_integrand * int_bg_term)
    prefactor = 1/(ν * φ^2 * (1+exp(2*φ*H)))
    tau_term  = τs(x, y) * (exp(φ*H) - 1)^2
    integral_term = sympy.integrate(tmp_B_integrand2, (ξ, -H, 0))
    tmp_B_fn = prefactor * (integral_term / Ek + tau_term)
    return tmp_B_fn
end
Latex_B_fn = make_Latex_B_fn(int_bg_term)
𝔘_tmp2 = 𝔘_tmp.subs(XXX, 0)      # remove the pressure-gradient term
testB = (𝔘_tmp2 - Latex_B_fn)
testB = testB.subs(bg(x, y, ξ), YYY)
testB = sympy.expand(testB).doit()
@assert testB == 0
println("2.7 LaTeX B(x) function matches.")
display(Latex_B_fn)

#### Check $G_{\mathfrak{u}} (z; -H)$

In [ ]:
Guv_ξ_eq_mH = sympy.simplify(Guv.subs(ξ, -H).args[1].args[1])
Latex_Guv_ξ_eq_mH = - sinh(φ * (z + H)) / (ν * φ)
@assert Guv_ξ_eq_mH == Latex_Guv_ξ_eq_mH
println("3.0 LaTeX Guv(z, -H) expression matches.")
println(Latex_Guv_ξ_eq_mH)
println()

### Make plots

In [ ]:
using CairoMakie

CairoMakie.activate!()

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
H_val = 1.0
f_val = 1.0
nu_constant = 1.0

# Choose Ekman numbers here
Ek1 = 0.25
Ek2 = 0.025
Ek_vals = [Ek1, Ek2]
varϕ_vals = [sqrt(1im * f_val / Ek) for Ek in Ek_vals]

zgrid = range(-H_val*0.999, 0.0, length=250)   # avoids undefined phase angle at z=-H_val

# Source depths for line cuts
# ξ_vals = [-0.9H, -0.6H, -0.3H, -0.1H, -0.001H]
ξ_vals = range(-H_val, 0, length=5)

# Colors for ξ families
ξcols = cgrad(:viridis, length(ξ_vals), categorical=true)

# -------------------------------------------------------------------
# Publication-style theme
# -------------------------------------------------------------------
set_theme!(
    Theme(
        fontsize=16,
        fonts=(; regular="TeX Gyre Heros"),
        Axis=(
            titlesize=20,
            xlabelsize=18,
            ylabelsize=18,
            xticklabelsize=16,
            yticklabelsize=16,
            xgridvisible=true,
            ygridvisible=true,
            xgridstyle=:dash,
            ygridstyle=:dash,
            xgridcolor=(:gray70, 0.35),
            ygridcolor=(:gray70, 0.35),
            spinewidth=1.2,
            xticksize=5,
            yticksize=5,
            xminorticksvisible=false,
            yminorticksvisible=false,
            topspinevisible=true,
            rightspinevisible=true,
        ),
        Legend=(
            framevisible=false,
            labelsize=15,
            titlesize=16,
            patchsize=(22, 14),
        ),
        Lines=(
            linewidth=2.5,
        )
    )
)

# -------------------------------------------------------------------
# Velocity Green's function G_u(z; ξ)
# -------------------------------------------------------------------
function G_u(z, ξ; H=H_val, varϕ=sqrt(1im * f_val / Ek))
    pref = exp(-varϕ * (z + ξ)) / (2 * nu_constant * varϕ * (1 + exp(2varϕ * H)))

    if z <= ξ
        return pref * (exp(2varϕ * ξ) - exp(2varϕ * (z + H)) - exp(2varϕ * (z + ξ + H)) + 1)
    else
        return pref * (exp(2varϕ * z) - exp(2varϕ * (ξ + H)) - exp(2varϕ * (z + ξ + H)) + 1)
    end
end

# -------------------------------------------------------------------
# Gu_int(z): integral of G_u over ξ
# -------------------------------------------------------------------
function Gu_int(z; H=H_val, varϕ=sqrt(1im * f_val / Ek))
    num = exp(varϕ*(z + H)) + exp(varϕ*(H - z)) - exp(2varϕ*H) - 1
    return num / (nu_constant * varϕ^2 * (1 + exp(2varϕ*H)))
end

# -------------------------------------------------------------------
# Styling helper
# -------------------------------------------------------------------
function style_axis!(ax)
    ax.topspinevisible = true
    ax.rightspinevisible = true
    ax.xgridvisible = true
    ax.ygridvisible = true
    ax.xgridstyle = :dash
    ax.ygridstyle = :dash
    ax.xgridcolor = (:gray70, 0.35)
    ax.ygridcolor = (:gray70, 0.35)
    return ax
end

# Colormap that fades from white to a given color
function fade_cmap(c)
    return cgrad([RGBAf(1, 1, 1, 0.12), c])
end

# -------------------------------------------------------------------
# Figure layout
# -------------------------------------------------------------------
fig = Figure(size=(1200, 1450))

Label(fig[0, 1:2],
    "Velocity Green's function and integrated kernel: magnitude and phase",
    fontsize=22,
    font=:bold
)

# -------------------------------------------------------------------
# Helper to plot one Ek row
# -------------------------------------------------------------------
function plot_ek_row!(fig, row, Ek, varϕ; H=H_val, zgrid=zgrid, ξvals=ξ_vals, ξcols=ξcols)

    panel_labels = row == 1 ? ("(a) ", "(b) ") : ("(c) ", "(d) ")

    ax_mag = Axis(fig[row, 1],
        xlabel="magnitude",
        ylabel="depth z/H",
        title="$(panel_labels[1])Ek = $(Ek): magnitude"
    )

    ax_ph = Axis(fig[row, 2],
        xlabel="phase [degrees]",
        ylabel="depth z/H",
        title="$(panel_labels[2])Ek = $(Ek): phase"
    )

    xlims!(ax_ph, -180, 180)

    # ----------------------------------------------------------------
    # G_u(z; ξ) family
    # ----------------------------------------------------------------
    for (j, ξ) in enumerate(ξvals)
        c = ξcols[j]
        Gvals = [G_u(z, ξ; H=H_val, varϕ=varϕ) for z in zgrid]

        mags = abs.(Gvals)
        phases = rad2deg.(angle.(Gvals))

        lines!(ax_mag, mags, zgrid,
            color=c,
            linewidth=2.4,
            label="G_u, ξ/H = $(round(ξ/H, digits=3))"
        )

        if isapprox(ξ, -H; atol=1e-12)
            phases = NaN .* phases
        end

        lines!(ax_ph, phases, zgrid,
            color=mags,
            colormap=fade_cmap(c),
            colorrange=(0, maximum(mags)),
            linewidth=2.4,
            label="G_u, ξ/H = $(round(ξ/H, digits=3))"
        )
    end

    # ----------------------------------------------------------------
    # Gu_int(z)
    # ----------------------------------------------------------------
    Gint_vals = [Gu_int(z; H=H_val, varϕ=varϕ) for z in zgrid]
    Gint_mag = abs.(Gint_vals)
    Gint_ph = rad2deg.(angle.(Gint_vals))

    lines!(ax_mag, Gint_mag, zgrid,
        color=:purple,
        linewidth=4.0,
        label="Gu_int"
    )

    lines!(ax_ph, Gint_ph, zgrid,
        color=Gint_mag,
        colormap=fade_cmap(:purple),
        colorrange=(0, maximum(Gint_mag)),
        linewidth=4.0,
        label="Gu_int"
    )

    for ax in (ax_mag, ax_ph)
        style_axis!(ax)
        ylims!(ax, -H, 0.0)
    end

    return ax_mag, ax_ph
end

# -------------------------------------------------------------------
# Plot both Ek rows
# -------------------------------------------------------------------
ax11, ax12 = plot_ek_row!(fig, 1, Ek_vals[1], varϕ_vals[1])
ax21, ax22 = plot_ek_row!(fig, 2, Ek_vals[2], varϕ_vals[2])

# Link y axes
linkyaxes!(ax11, ax21)
linkyaxes!(ax12, ax22)

# -------------------------------------------------------------------
# Single shared legend below the panels
# -------------------------------------------------------------------
legend_elements = LineElement[]
legend_labels = String[]

for (j, ξ) in enumerate(ξ_vals)
    push!(legend_elements, LineElement(color=ξcols[j], linewidth=3))
    push!(legend_labels, "ξ/H = $(round(ξ/H_val, digits=2))")
end

push!(legend_elements, LineElement(color=:purple, linewidth=3))
push!(legend_labels, "Gu_int")

legend_g = Legend(fig[3, 1:2],
    legend_elements,
    legend_labels,
    title="Green's function panel",
    orientation=:horizontal,
    nbanks=2,
    framevisible=false,
    tellwidth=false,
    halign=:center
)


# -------------------------------------------------------------------
# Tightened spacing
# -------------------------------------------------------------------
colgap!(fig.layout, 1, 10)
rowgap!(fig.layout, 1, 4)
rowgap!(fig.layout, 2, 2)

# -------------------------------------------------------------------
# Save to PDF
# -------------------------------------------------------------------
CairoMakie.save("velocity_Gu_magnitude_phase_two_Ek.pdf", fig)

fig

In [ ]:
println("Surface velocity orientation wrt stress with zero pressure gradient forcing:")
surface_stress = 1 + 0.0im
println("Surface stress = $(surface_stress)")

for k in eachindex(Ek_vals)
    surface_velocity = -G_u(0.0, 0.0; H=H_val, varϕ=varϕ_vals[k]) * surface_stress

    surface_velocity_orientation = angle(surface_velocity / surface_stress)
    println("Ek = $(Ek_vals[k]): surface velocity orientation = $(round(rad2deg(surface_velocity_orientation), digits=2)) degrees")
end

println("\nSurface velocity orientation wrt pressure gradient with zero stress forcing:")
surface_pressure_gradient = 1 + 0.0im
println("Surface pressure gradient = $(surface_pressure_gradient)")
for k in eachindex(Ek_vals)
    surface_velocity = Gu_int(0.0; H=H_val, varϕ=varϕ_vals[k]) * surface_pressure_gradient

    surface_velocity_orientation = angle(surface_velocity / surface_pressure_gradient)
    println("Ek = $(Ek_vals[k]): surface velocity orientation = $(round(rad2deg(surface_velocity_orientation), digits=2)) degrees")
end

println("\nBottom stress orientation wrt surface velocity with zero stress forcing:")
for k in eachindex(Ek_vals)
    surface_velocity = Gu_int(0.0; H=H_val, varϕ=varϕ_vals[k]) * surface_pressure_gradient
    bottom_stress = (1/varϕ_vals[k]) * tanh(varϕ_vals[k] * H_val) * surface_pressure_gradient

    bottom_stress_orientation = angle(bottom_stress / surface_velocity)
    println("Ek = $(Ek_vals[k]): bottom stress orientation = $(round(rad2deg(bottom_stress_orientation), digits=2)) degrees")
end

println("\nPositive angles are counterclockwise from the forcing direction, negative angles are clockwise.")